# Coffee17 KD Teacher Diversity — 7-Rotation Audit\n\nAudit diagnostik saja. **Tidak melatih student dan tidak membuka outer test.**\n\nUntuk setiap fold, semua training image dievaluasi dengan teacher R0/C0/F0/W0 pada rotasi `0,45,90,135,180,225,270` derajat. Notebook juga merekonstruksi jadwal rotasi deterministik 50 epoch yang benar-benar digunakan oleh training loader, sehingga kita dapat mengukur seberapa sering student seharusnya melihat teacher disagreement.\n\nSebelum Run All:\n1. Add Input dataset Coffee17 original.\n2. Add Input saved primary preprocessing output yang berisi `coffee17-preprocessing-project`.\n3. Aktifkan GPU + Internet.\n

In [ ]:
# Coffee17 KD teacher diversity — seven-angle audit only
SCIENTIFIC_CODE_COMMIT = "4013314e4bcc0d10a2f22d66e2e84c5aa953e631"

import hashlib, importlib, json, os, shutil, subprocess, sys, zipfile
from pathlib import Path

INPUT = Path("/kaggle/input")
WORK = Path("/kaggle/working")
REPO = WORK / "coffee-bean-classification-rotation-audit-code"
PROJECT = WORK / "coffee17-preprocessing-kd-rotation-audit-project"

assert INPUT.is_dir() and WORK.is_dir(), "Notebook ini harus dijalankan di Kaggle."

def sha256_file(path):
    h = hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda: f.read(1024 * 1024), b""):
            h.update(block)
    return h.hexdigest()

def merge_tree_exact(source, target):
    source, target = Path(source), Path(target)
    for item in sorted(source.rglob("*")):
        if not item.is_file():
            continue
        rel = item.relative_to(source)
        dst = target / rel
        dst.parent.mkdir(parents=True, exist_ok=True)
        if dst.is_file():
            if sha256_file(item) != sha256_file(dst):
                raise RuntimeError(f"Kaggle input conflict: {rel}")
        else:
            shutil.copy2(item, dst)

def run(command, cwd=None, log_path=None):
    command = [str(x) for x in command]
    print("\n$ " + " ".join(command), flush=True)
    if log_path is None:
        subprocess.run(command, cwd=cwd, check=True)
        return
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    with log_path.open("a", encoding="utf-8") as stream:
        process = subprocess.Popen(
            command,
            cwd=cwd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        assert process.stdout is not None
        for line in process.stdout:
            print(line, end="")
            stream.write(line)
            stream.flush()
        rc = process.wait()
    if rc:
        raise RuntimeError(f"Command gagal ({rc}): {' '.join(command)}")

# ------------------------------------------------------------
# 1. Locate frozen primary output containing the 20 teachers.
# ------------------------------------------------------------
primary_candidates = []
for p in INPUT.rglob("coffee17-preprocessing-project"):
    if not p.is_dir():
        continue
    authority = p / "evidence/coffee17-preprocessing-primary-v1/preprocessing_primary_confirmation.json"
    experiments = p / "experiments/coffee17-preprocessing-primary-v1"
    clean_manifest = p / "evidence/coffee17-preprocessing-data-v1/clean_manifest.json"
    fold_manifest = p / "evidence/coffee17-preprocessing-data-v1/fold_manifest.json"
    if all(x.is_file() for x in (authority, clean_manifest, fold_manifest)) and experiments.is_dir():
        payload = json.loads(authority.read_text(encoding="utf-8"))
        if payload.get("decision") == "AUTHORIZE_OOF_TEST_EVALUATION" and payload.get("completed_runs") == 20:
            primary_candidates.append(p)

if not primary_candidates:
    raise FileNotFoundError(
        "Saved primary output tidak ditemukan. Add Input -> saved output notebook preprocessing lama."
    )

authority_hashes = {
    sha256_file(
        p / "evidence/coffee17-preprocessing-primary-v1/preprocessing_primary_confirmation.json"
    )
    for p in primary_candidates
}
if len(authority_hashes) != 1:
    raise RuntimeError("Ada beberapa primary outputs dengan authority berbeda.")

PRIMARY_PROJECT = sorted(primary_candidates, key=lambda p: str(p))[0]
print("PRIMARY PROJECT:", PRIMARY_PROJECT)

DATA_EVIDENCE = PRIMARY_PROJECT / "evidence/coffee17-preprocessing-data-v1"
AUTHORITY = PRIMARY_PROJECT / "evidence/coffee17-preprocessing-primary-v1/preprocessing_primary_confirmation.json"
PRIMARY_EXPERIMENTS = PRIMARY_PROJECT / "experiments/coffee17-preprocessing-primary-v1"

# Resume optional if a previous rotation-audit output is attached.
for prior in sorted(
    p for p in INPUT.rglob("coffee17-preprocessing-kd-rotation-audit-project")
    if p.is_dir()
):
    print("MERGE PRIOR ROTATION AUDIT:", prior)
    merge_tree_exact(prior, PROJECT)

PROJECT.mkdir(parents=True, exist_ok=True)
(PROJECT / "folds").mkdir(parents=True, exist_ok=True)
(PROJECT / "logs").mkdir(parents=True, exist_ok=True)
(PROJECT / "analysis").mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# 2. Exact scientific code + frozen preprocessing environment.
# ------------------------------------------------------------
if REPO.exists():
    shutil.rmtree(REPO)
run([
    "git", "clone", "--quiet", "--no-checkout",
    "https://github.com/ediprin/coffee-bean-classification.git", REPO
])
run(["git", "-C", REPO, "checkout", "--quiet", "--detach", SCIENTIFIC_CODE_COMMIT])

prior_lock = PRIMARY_PROJECT / "evidence/coffee17-preprocessing-runtime-v1/requirements_preprocessing_study_lock.txt"
requirements = prior_lock if prior_lock.is_file() else REPO / "requirements/preprocessing-study.txt"
run([sys.executable, "-m", "pip", "install", "-q", "-r", requirements])
run([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", REPO])
sys.path.insert(0, str(REPO / "src"))
importlib.invalidate_caches()
os.chdir(REPO)

import torch
if not torch.cuda.is_available():
    raise RuntimeError("Aktifkan Kaggle GPU.")
print("GPU:", torch.cuda.get_device_name(0))

from bilinear_lmmd.data.preparation.prepare_coffee17 import discover_directory_samples
from bilinear_lmmd.data.preparation.audit_coffee17_provenance import audit_coffee17_provenance
from bilinear_lmmd.data.preparation.prepare_preprocessing_folds import prepare_preprocessing_folds
from bilinear_lmmd.data.preparation.materialize_preprocessing_development import (
    materialize_preprocessing_development,
)

# ------------------------------------------------------------
# 3. Reconstruct exact frozen data/folds. No model training.
# ------------------------------------------------------------
print("\n=== RECONSTRUCT + VERIFY COFFEE17 ===")
by_class = discover_directory_samples(INPUT)
raw_count = sum(len(v) for v in by_class.values())
print("Coffee17 mounted:", raw_count, "images /", len(by_class), "classes")
if raw_count != 979 or len(by_class) != 17:
    raise RuntimeError(f"Coffee17 raw population tidak sesuai: {raw_count}/17.")

ARCHIVE = WORK / "coffee17_rotation_audit_original.zip"
PROV_LOCAL = WORK / "coffee17_rotation_audit_provenance"
CANONICAL = WORK / "coffee17_rotation_audit_original_v1"
FOLDS_LOCAL = WORK / "coffee17_rotation_audit_folds"

for path in (PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()

with zipfile.ZipFile(ARCHIVE, "w", compression=zipfile.ZIP_STORED) as bundle:
    for class_name, paths in sorted(by_class.items()):
        for path in sorted(paths):
            info = zipfile.ZipInfo(
                f"{class_name}/{path.name}",
                date_time=(1980, 1, 1, 0, 0, 0),
            )
            info.compress_type = zipfile.ZIP_STORED
            info.external_attr = 0o644 << 16
            bundle.writestr(info, path.read_bytes())

provenance = audit_coffee17_provenance(
    ARCHIVE, PROV_LOCAL, canonical_root=CANONICAL
)
if provenance["decision"] != "PASS":
    raise RuntimeError(f"Provenance gagal: {provenance['decision']}")

fold_summary = prepare_preprocessing_folds(
    CANONICAL,
    PROV_LOCAL / "coffee17_provenance.json",
    FOLDS_LOCAL,
    folds=5,
    seed=42,
    validation_ratio=0.10,
)
if fold_summary["decision"] != "PASS_COFFEE17_PREPROCESSING_DATA_GATE":
    raise RuntimeError("Fold reconstruction gagal.")

for name in ("clean_manifest.json", "fold_manifest.json"):
    reconstructed = FOLDS_LOCAL / name
    frozen = DATA_EVIDENCE / name
    if sha256_file(reconstructed) != sha256_file(frozen):
        raise RuntimeError(f"Reconstructed {name} berbeda dari frozen primary evidence.")

print("DATA/FOLD HASH MATCH: PASS")

# ------------------------------------------------------------
# 4. Seven-angle audit on each fold's TRAIN split only.
# ------------------------------------------------------------
for fold in range(1, 6):
    print(f"\n================ ROTATION AUDIT FOLD {fold}/5 ================", flush=True)

    out_dir = PROJECT / "folds" / f"fold_{fold}"
    result = out_dir / "rotation_audit.json"
    if result.is_file():
        print("REUSE:", result)
        continue

    DEV = WORK / f"coffee17_rotation_audit_dev_fold_{fold}"
    shutil.rmtree(DEV, ignore_errors=True)
    materialize_preprocessing_development(
        CANONICAL,
        DATA_EVIDENCE / "clean_manifest.json",
        DATA_EVIDENCE / "fold_manifest.json",
        DEV,
        fold=fold,
    )

    run([
        sys.executable, "-u", "-m",
        "bilinear_lmmd.experiments.run_preprocessing_kd_rotation_audit",
        "--data-root", DEV,
        "--authority", AUTHORITY,
        "--experiments-root", PRIMARY_EXPERIMENTS,
        "--fold", str(fold),
        "--output-dir", out_dir,
        "--device", "cuda:0",
        "--image-size", "224",
        "--batch-size", "32",
        "--workers", "4",
    ], cwd=REPO, log_path=PROJECT / "logs" / f"rotation_audit_fold{fold}.log")

    shutil.rmtree(DEV, ignore_errors=True)
    torch.cuda.empty_cache()

# ------------------------------------------------------------
# 5. Fold summary. Still no outer OOF/test access.
# ------------------------------------------------------------
SUMMARY = PROJECT / "analysis" / "rotation_audit_summary.json"
run([
    sys.executable, "-u", "-m",
    "bilinear_lmmd.experiments.run_preprocessing_kd_rotation_audit_summary",
    "--audit-root", PROJECT / "folds",
    "--output", SUMMARY,
], cwd=REPO)

summary = json.loads(SUMMARY.read_text(encoding="utf-8"))
fm = summary["fold_mean"]

print("\n================ KEY RESULT ================")
print(
    "0° teacher disagreement        :",
    f"{fm['zero_angle_disagreement']['mean']:.2%}"
)
print(
    "Any disagreement across 7     :",
    f"{fm['any_disagreement_in_7_angles']['mean']:.2%}"
)
print(
    "Scheduled 50-epoch exposure   :",
    f"{fm['scheduled_disagreement_exposure']['mean']:.2%}"
)
print(
    "Images exposed >=1 epoch      :",
    f"{fm['scheduled_images_with_at_least_one_disagreement_epoch']['mean']:.2%}"
)
print(
    "0° mean pairwise JS           :",
    f"{fm['zero_angle_mean_pairwise_js']['mean']:.6f}"
)
print(
    "Scheduled mean pairwise JS    :",
    f"{fm['scheduled_mean_pairwise_js']['mean']:.6f}"
)

# ------------------------------------------------------------
# 6. Small ZIP for ChatGPT analysis.
# ------------------------------------------------------------
ZIP_BASE = WORK / "rotation-audit-package"
zip_path = Path(
    shutil.make_archive(
        str(ZIP_BASE),
        "zip",
        root_dir=PROJECT,
    )
)
print("\nREADY:", zip_path)
print("SIZE :", round(zip_path.stat().st_size / 1024 / 1024, 2), "MB")
print("Tidak ada student training dan tidak ada outer-test inference.")

# Clean temporary large files, retain PROJECT + ZIP.
for path in (REPO, PROV_LOCAL, CANONICAL, FOLDS_LOCAL):
    shutil.rmtree(path, ignore_errors=True)
if ARCHIVE.exists():
    ARCHIVE.unlink()
